In [23]:
%pip install numpy pandas

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [24]:
import json
import re
import numpy as np
import pandas as pd

What variable we want to explore the correlation with single or multi listing

estimated_revenue_l365d
estimated_occupancy_l365d
number_of_reviews_ly
price
...

In [37]:
listings_raw = pd.read_csv('listings_raw.csv')
reviews_raw = pd.read_csv('reviews_raw.csv')

listings_raw.head(5)

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,10803,https://www.airbnb.com/rooms/10803,20260616211523,2026-06-17,city scrape,"Room in Deco Apartment, Brunswick East",A large air conditioned room with firm queen s...,NaN,https://a0.muscache.com/pictures/e5f30dd1-ac57...,38901,...,4.73,4.70,4.66,NaN,NaN,1,0,1,0,1.31
1,12936,https://www.airbnb.com/rooms/12936,20260616211523,2026-06-28,previous scrape,St Kilda 1BR+BEACHSIDE+BALCONY+WIFI+AC,RIGHT IN THE HEART OF ST KILDA! It doesn't get...,NaN,https://a0.muscache.com/pictures/59701/2e8cdaf...,50121,...,4.83,4.78,4.66,NaN,NaN,10,10,0,0,0.22
2,41836,https://www.airbnb.com/rooms/41836,20260616211523,2026-06-28,previous scrape,CLOSE TO CITY & MELBOURNE AIRPORT,Easy to travel from and to the Airport; quiet ...,NaN,https://a0.muscache.com/pictures/569696dd-1ad0...,182833,...,4.83,4.39,4.69,NaN,NaN,2,0,2,0,0.83
3,43429,https://www.airbnb.com/rooms/43429,20260616211523,2026-06-17,city scrape,Tranquil Javanese Studio and Pond!,"No service/Cleaning Fees, EV Charger, Study th...",NaN,https://a0.muscache.com/pictures/airflow/Hosti...,189684,...,4.94,4.79,4.86,NaN,NaN,2,2,0,0,1.48
4,44699,https://www.airbnb.com/rooms/44699,20260616211523,2026-06-17,city scrape,"15 yearsHosting (4.8), 8 CITY TRAMS, GymPoolTe...",Unwavering service — just ask and we do our be...,NaN,https://a0.muscache.com/pictures/miso/Hosting-...,189245,...,4.97,4.84,4.71,NaN,NaN,1,0,1,0,0.35


In [39]:
print(listings_raw.shape)
print(reviews_raw.shape)

(25728, 90)
(1026690, 6)


clean price into int for comparison

In [27]:
def parse_price(df):
    df = df.copy()
    
    def parse_price_int(price):
        """
        Independently re-implement the parsing logic in a different way 
        """
        if pd.isna(price):
            return np.nan

        clean_price = str(price).replace("$", "").replace(",", "").strip()
        try:
            return float(clean_price)
        except:
            return np.nan

    df['price_int'] = df['price'].apply(parse_price_int)

    return df

turn the number of listings into only single or multilisting

In [28]:
def listing_class(df):
    df = df.copy()
    
    def listing_class_logic(num_listings):

        if pd.isna(num_listings):
            return np.nan
        else:
            num_listings = int(num_listings)
        
        if num_listings <= 0:
            return np.nan
        elif num_listings == 1:
            return "Singlelisting"
        else:
            return "Multilisting"

    df['listing_class'] = df['calculated_host_listings_count'].apply(listing_class_logic)

    return df

# EDA

## Before Preprocessing

In [40]:
def column_audit(df):
    rows = len(df)

    result = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_missing": df.notna().sum(),
        "missing_count": df.isna().sum(),
        "missing_percent": df.isna().mean().mul(100),
        "unique_count": df.nunique(dropna=True)
    })

    result["unique_percent"] = (
        result["unique_count"] / rows * 100
    )

    return result.sort_values(
        ["missing_percent", "unique_count"],
        ascending=[False, True]
    )

In [41]:
listings_audit = column_audit(listings_raw)
reviews_audit = column_audit(reviews_raw)

display(listings_audit)
display(reviews_audit)

,dtype,non_missing,missing_count,missing_percent,unique_count,unique_percent
neighborhood_overview,float64,0,25728,100.0,0,0.000000
host_since,float64,0,25728,100.0,0,0.000000
host_response_time,float64,0,25728,100.0,0,0.000000
host_response_rate,float64,0,25728,100.0,0,0.000000
host_acceptance_rate,float64,0,25728,100.0,0,0.000000
...,...,...,...,...,...,...
amenities,object,25728,0,0.0,24395,94.818874
name,object,25728,0,0.0,25021,97.252021
picture_url,object,25728,0,0.0,25196,97.932214
id,int64,25728,0,0.0,25728,100.000000


,dtype,non_missing,missing_count,missing_percent,unique_count,unique_percent
comments,object,1026557,133,0.012954,979476,95.401338
reviewer_name,object,1026689,1,0.000097,121850,11.868237
date,object,1026690,0,0.000000,5197,0.506190
listing_id,int64,1026690,0,0.000000,21251,2.069856
reviewer_id,int64,1026690,0,0.000000,794203,77.355677
id,int64,1026690,0,0.000000,1026690,100.000000


In [31]:
listings_all_missing =  listings_raw.columns[
    listings_raw.isna().all()
].tolist()

reviews_all_missing = reviews_raw.columns[
    reviews_raw.isna().all()
].tolist()

print("Listings columns with 100% missing:")
print(listings_all_missing)

print("Reviews columns with 100% missing:")
print(reviews_all_missing)

Listings columns with 100% missing:
['neighborhood_overview', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_thumbnail_url', 'host_neighbourhood', 'host_total_listings_count', 'host_verifications', 'neighbourhood', 'neighbourhood_group_cleansed', 'calendar_updated', 'license', 'instant_bookable']
Reviews columns with 100% missing:
[]


In [42]:
listings_work = listings_raw.drop(
    columns=listings_all_missing
).copy()

reviews_work = reviews_raw.copy()

print("Listings before:", listings_raw.shape)
print("Listings after:", listings_work.shape)

print("Reviews before:", reviews_raw.shape)
print("Reviews after:", reviews_work.shape)

Listings before: (25728, 90)
Listings after: (25728, 76)
Reviews before: (1026690, 6)
Reviews after: (1026690, 6)
